# 04 - Train ResNet18 SimCLR Experiments

This notebook runs the four ResNet18 contrastive experiments from the report:

- `resnet18_covidqu`
- `resnet18_imagenet_covidqu`
- `resnet18_covidqu_syn`
- `resnet18_imagenet_covidqu_syn`

It calls Python scripts only. SimCLR pretraining and supervised fine-tuning logic live in the repo scripts.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull Repo

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('Repo:', Path.cwd())

## 3. Install Minimal Dependencies

Colab already provides PyTorch and torchvision. Install only lightweight packages needed by the scripts.

In [ ]:
!pip install -q scikit-learn matplotlib pandas Pillow PyYAML

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Editable Runner Variables

`SYNTHETIC_MANIFEST` should point to the Stage 1 DCGAN manifest generated on Drive. It is only used by the `COVID-QU-Syn` experiments.

In [ ]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/contrastive-synthesis-medcls_CVProject/results/experiments')
SYNTHETIC_MANIFEST = Path('/content/drive/MyDrive/path/to/synthetic_dcgan.csv')

RUN_RESNET_COVIDQU = True
RUN_RESNET_IMAGENET_COVIDQU = True
RUN_RESNET_COVIDQU_SYN = True
RUN_RESNET_IMAGENET_COVIDQU_SYN = True

# Use 1 for a smoke test, or None to use config epochs.
EPOCH_OVERRIDE = None

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST)

## 5. Lightweight Checks

In [ ]:
!python scripts/check_experiment_inputs.py
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py

## 6. Helper For Optional Epoch Override

In [ ]:
epoch_arg = '' if EPOCH_OVERRIDE is None else f'--epochs {EPOCH_OVERRIDE}'
print('epoch_arg:', epoch_arg)

## 7. Experiment: resnet18_covidqu

SimCLR pretraining from random initialization on real unlabeled COVID-QU, then supervised fine-tuning on real labeled manifests.

In [ ]:
EXP = 'resnet18_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

if RUN_RESNET_COVIDQU:
    !python scripts/run_simclr_resnet.py \
      --config configs/experiments/resnet18/covidqu.yaml \
      --output-dir "{OUT}" \
      {epoch_arg}
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/covidqu.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {epoch_arg}
else:
    print('Skipping', EXP)

## 8. Experiment: resnet18_imagenet_covidqu

SimCLR pretraining from ImageNet initialization on real unlabeled COVID-QU, then supervised fine-tuning on real labeled manifests.

In [ ]:
EXP = 'resnet18_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

if RUN_RESNET_IMAGENET_COVIDQU:
    !python scripts/run_simclr_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu.yaml \
      --output-dir "{OUT}" \
      {epoch_arg}
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {epoch_arg}
else:
    print('Skipping', EXP)

## 9. Experiment: resnet18_covidqu_syn

SimCLR pretraining from random initialization on Stage 1 DCGAN synthetic images, then supervised fine-tuning on real labeled manifests.

In [ ]:
EXP = 'resnet18_covidqu_syn'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

if RUN_RESNET_COVIDQU_SYN:
    !python scripts/run_simclr_resnet.py \
      --config configs/experiments/resnet18/covidqu_syn.yaml \
      --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
      --output-dir "{OUT}" \
      {epoch_arg}
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/covidqu_syn.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {epoch_arg}
else:
    print('Skipping', EXP)

## 10. Experiment: resnet18_imagenet_covidqu_syn

SimCLR pretraining from ImageNet initialization on Stage 1 DCGAN synthetic images, then supervised fine-tuning on real labeled manifests.

In [ ]:
EXP = 'resnet18_imagenet_covidqu_syn'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

if RUN_RESNET_IMAGENET_COVIDQU_SYN:
    !python scripts/run_simclr_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu_syn.yaml \
      --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
      --output-dir "{OUT}" \
      {epoch_arg}
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu_syn.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {epoch_arg}
else:
    print('Skipping', EXP)

## 11. Display Result Table

In [ ]:
import json
import pandas as pd

rows = []
for metrics_path in sorted(OUTPUT_ROOT.glob('resnet18_*/metrics.json')):
    row = {'experiment_id': metrics_path.parent.name}
    row.update(json.loads(metrics_path.read_text()))
    rows.append(row)

display(pd.DataFrame(rows))
!find "{OUTPUT_ROOT}" -maxdepth 2 -type f | sort